# IVPM-EW: Simulation-Calibrated Multi-Platform Early-Warning System for Viral Ignition
## Full experiment suite (companion notebook of the IEEE Access article)

**Setup:**
1. Upload the two CSV files (`FINAL_ACADEMIC_DATASET_SIMGE.csv`, `FINAL_ACADEMIC_DATASET_KATE_BUSH.csv`) as a Kaggle Dataset and attach it to this notebook via **Add Input**.
2. **No GPU is required.** The workload is CPU-parallel; the notebook uses all available cores through `joblib`. Accelerator can be set to *None*.
3. Select *Run All*. Estimated runtime: 35-60 minutes in total. All outputs are written to `/kaggle/working/` and bundled into `ivpm_ew_results_v2.zip` at the end.

**Experiments:**
- EXP1: Threshold x confirmation x persistence grid (operating characteristics)
- EXP2: Censoring ablation (why the 2-of-3 rule is a structural necessity)
- EXP3: Detector comparison (persistent exceedance vs. CUSUM vs. causal CP-PSI composite)
- EXP4: Regime classification: horizon sweep and sim-to-real transfer
- EXP5: Zero-shot validation on the two real archetypes (Simge / Kate Bush)
- EXP6: Honest forecasting benchmark (expanding window, naive persistence included)
- EXP4b + EXP7: delta-jump baseline and five-seed statistical replication


In [ ]:
# === Cell 1: common setup ===
import os, glob, json, warnings, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")
plt.rcParams.update({"font.size": 11, "font.family": "serif",
                     "axes.grid": True, "grid.alpha": 0.3})
N_JOBS = os.cpu_count() or 2
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./out"
os.makedirs(OUT, exist_ok=True)
T = 72
print(f"CPU cores: {N_JOBS} | output directory: {OUT}")

In [ ]:
# === Cell 2: core module — generator (v2), detectors, classifiers ===

def _ar1_noise(n, rng, rho=0.5, sigma=0.15):
    e = np.zeros(n)
    for t in range(1, n):
        e[t] = rho * e[t-1] + rng.normal(0, sigma)
    return e

def generate_series(typology, rng, censor=True, burst_prob=0.5):
    """typology in {'null','endogenous','exogenous'}; log10 scale.
    v2: a fraction of endogenous series is burst-initiated (Simge archetype)."""
    base = rng.uniform(1.0, 3.0)
    onset = int(rng.integers(18, T-24))
    lead_yt, lead_tr = int(rng.integers(1,4)), int(rng.integers(1,3))
    L = np.full(T, base, float)
    if typology == "endogenous":
        amp = rng.uniform(1.0, 2.0); k = rng.uniform(0.6, 1.4)
        ramp = amp/(1+np.exp(-k*(np.arange(T)-onset-6)))
        ds = onset + int(rng.integers(8,14)); d = rng.uniform(0.06,0.12)
        plateau = rng.uniform(0.35,0.6); decay = np.ones(T)
        for t in range(ds, T):
            decay[t] = plateau + (1-plateau)*np.exp(-d*(t-ds))
        L = base + ramp*decay
    elif typology == "exogenous":
        amp = rng.uniform(1.5,2.5); d = rng.uniform(0.25,0.5)
        for t in range(onset, T):
            L[t] = base + amp*np.exp(-d*(t-onset))
    sy = lead_yt if typology=="endogenous" else 0
    st = lead_tr if typology=="endogenous" else 0
    L_yt = np.concatenate([L[sy:], np.full(sy, L[-1])])
    L_tr = np.concatenate([L[st:], np.full(st, L[-1])])
    yt = L_yt + _ar1_noise(T, rng)
    tr0 = L_tr + _ar1_noise(T, rng)
    spu = L + _ar1_noise(T, rng)
    if typology=="endogenous" and rng.random() < burst_prob:
        # burst-initiated endogenous: 30-70% of eventual amplitude within 2 months
        for arr in (yt, tr0, spu):
            b = arr[:onset].mean(); amp2 = arr[onset:].max()-b
            frac = rng.uniform(0.3, 0.7)
            for t in range(onset, min(onset+2, T)):
                arr[t] = max(arr[t], b + frac*amp2 + rng.normal(0, 0.05))
    if censor:
        c = base + rng.uniform(0.5, 0.9)
        sp = np.where(spu >= c, spu, 0.0)
    else:
        sp = spu
    lin = 10**tr0
    tr = np.log10(np.maximum(1, np.round(100*lin/lin.max())))
    return dict(typology=typology, onset=onset, sp=sp, yt=yt, tr=tr,
                sp_uncensored=spu)

# ---------- causal flags ----------
def causal_flags(x, thr, burn_in=12, censored=False, persist=2):
    x = np.asarray(x, float); raw = np.zeros(len(x), bool)
    for t in range(burn_in, len(x)):
        past = x[:t]
        if censored:
            past = past[past > 0]
            if len(past) < 6 or x[t] == 0: continue
        sd = past.std() or 1e-6
        if x[t] > past.mean() + thr*sd: raw[t] = True
    out = np.zeros_like(raw)
    for t in range(persist-1, len(raw)):
        if raw[max(0,t-persist+1):t+1].sum() >= persist: out[t] = True
    return out

def cusum_flags(x, k=0.5, h=6.0, burn_in=12, censored=False):
    x = np.asarray(x, float); Cst = 0.0; out = np.zeros(len(x), bool)
    for t in range(burn_in, len(x)):
        past = x[:t]
        if censored:
            past = past[past > 0]
            if len(past) < 6 or x[t] == 0: continue
        sd = past.std() or 1e-6
        Cst = max(0.0, Cst + (x[t]-past.mean())/sd - k)
        if Cst > h: out[t] = True
    return out

def composite_flags(s, thr, burn_in=12, persist=2):
    """Causal CP-PSI: mean of the three expanding-window z-scores (single series)."""
    n = len(s["yt"]); comp = np.zeros(n)
    for t in range(burn_in, n):
        zs = []
        for x, cens in [(s["yt"],False),(s["tr"],False),(s["sp"],True)]:
            past = x[:t]
            if cens:
                past = past[past>0]
                if len(past)<6 or x[t]==0: continue
            sd = past.std() or 1e-6
            zs.append((x[t]-past.mean())/sd)
        comp[t] = np.mean(zs) if zs else 0.0
    raw = comp > thr
    out = np.zeros_like(raw)
    for t in range(persist-1, n):
        if raw[max(0,t-persist+1):t+1].sum() >= persist: out[t] = True
    return out

def first_confirmed(allf, confirm, window=2):
    for t in range(allf.shape[1]):
        lo = max(0, t-window+1)
        if allf[:, lo:t+1].any(axis=1).sum() >= confirm: return t
    return None

def detect_pe(s, thr=1.75, confirm=2, window=2, persist=2):
    allf = np.vstack([causal_flags(s["yt"],thr,persist=persist),
                      causal_flags(s["tr"],thr,persist=persist),
                      causal_flags(s["sp"],thr,censored=True,persist=persist)])
    return first_confirmed(allf, confirm, window)

def detect_cusum(s, k=0.5, h=6.0, confirm=2, window=2):
    allf = np.vstack([cusum_flags(s["yt"],k,h), cusum_flags(s["tr"],k,h),
                      cusum_flags(s["sp"],k,h,censored=True)])
    return first_confirmed(allf, confirm, window)

def detect_composite(s, thr=1.75, persist=2):
    f = composite_flags(s, thr, persist=persist)
    return int(f.argmax()) if f.any() else None

# ---------- regime classification ----------
def rise_ratio(yt, ign, h=12):
    b = yt[max(0,ign-6):ign].mean()
    peak = yt[ign:min(ign+h+1,len(yt))].max()
    return (yt[ign]-b)/max(peak-b, 1e-6)

def causal_features(s, ign, h):
    yt, tr, sp = s["yt"], s["tr"], s["sp"]
    b = yt[max(0,ign-6):ign].mean()
    w = yt[ign:min(ign+h+1,len(yt))]
    def jz(x, c=False):
        p = x[:ign]; p = p[p>0] if c else p
        return abs(x[ign]-p[-1])/(p.std() or 1e-6) if len(p)>5 else 0.0
    trb = tr[max(0,ign-6):ign].mean()
    tramp = max(tr[ign:min(ign+h+1,len(tr))].max()-trb, 1e-6)
    return [rise_ratio(yt,ign,h), (tr[ign]-trb)/tramp,
            (w.max()-yt[ign])/max(w.max()-b,1e-6),
            np.argmax(w)/max(h,1),
            (w[-1]-w.max())/max(w.max()-b,1e-6),
            jz(yt), jz(tr), jz(sp,True),
            yt[max(0,ign-12):ign].std()]
print("core module loaded")

## EXP1 — Operating-characteristic grid (persistent-exceedance detector)

In [ ]:
# === Cell 3: EXP1 ===
def run_cell(thr, confirm, persist, censor=True, n_per_class=300, seed=7,
             detector="pe", cusum_k=0.5, cusum_h=6.0):
    rng = np.random.default_rng(seed)
    rows = []
    for typ in ["null","endogenous","exogenous"]:
        for _ in range(n_per_class):
            s = generate_series(typ, rng, censor=censor)
            if detector == "pe":
                ign = detect_pe(s, thr, confirm, persist=persist)
            elif detector == "cusum":
                ign = detect_cusum(s, k=cusum_k, h=cusum_h, confirm=confirm)
            else:
                ign = detect_composite(s, thr, persist=persist)
            valid = ign is not None and typ != "null" and ign >= s["onset"]-3
            regime = None
            if valid:
                regime = "exogenous" if rise_ratio(s["yt"], ign, 12) >= 0.6 \
                         else "endogenous"
            rows.append(dict(truth=typ, det=ign is not None, valid=valid,
                             prem=(ign is not None and typ!="null"
                                   and ign < s["onset"]-3),
                             regime=regime,
                             delay=(ign-s["onset"]) if valid else np.nan,
                             lead=(int(np.argmax(s["sp"]))-ign) if valid else np.nan))
    df = pd.DataFrame(rows)
    ev, nu = df[df.truth!="null"], df[df.truth=="null"]
    ok = ev[ev.valid]
    return dict(detector=detector, thr=thr, confirm=confirm, persist=persist,
                censor=censor, TPR=100*ev.valid.mean(),
                premature=100*ev.prem.mean(), FPR=100*nu.det.mean(),
                delay=ok.delay.mean(), lead=ok.lead.mean(),
                regime_acc=100*(ok.regime==ok.truth).mean())

grid_cfg = [(thr,c,p) for thr in [1.25,1.5,1.75,2.0,2.25,2.5]
            for c in [1,2,3] for p in [1,2]]
grid = Parallel(n_jobs=N_JOBS)(delayed(run_cell)(thr,c,p) for thr,c,p in grid_cfg)
g = pd.DataFrame(grid).round(2)
g.to_csv(f"{OUT}/results_grid.csv", index=False)
print(g.to_string(index=False))

In [ ]:
# === Cell 4: EXP1 figuru ===
g2 = g[g.confirm==2]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for p, m in [(1,"o"),(2,"s")]:
    d = g2[g2.persist==p].sort_values("thr")
    axes[0].plot(d.thr, d.TPR, marker=m, label=f"persistence={p}")
    axes[1].plot(d.thr, d.FPR, marker=m, label=f"persistence={p}")
    axes[2].plot(d.thr, d.delay, marker=m, label=f"persistence={p}")
for ax, yl in zip(axes, ["Valid TPR (%)","False-alarm rate (%)",
                          "Mean detection delay (mo)"]):
    ax.set_xlabel(r"threshold $\theta$ ($\sigma$)"); ax.set_ylabel(yl)
    ax.legend(fontsize=9)
axes[0].axhline(95, color="gray", ls=":", lw=1)
fig.suptitle("2-of-3 rule: operating characteristics (N=900/cell)")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_operating_curve.png", dpi=200)
plt.show()

## EXP2 — Censoring ablation: why 2-of-3 is a structural necessity

In [ ]:
# === Cell 5: EXP2 ===
abl = Parallel(n_jobs=N_JOBS)(delayed(run_cell)(1.75, c, 2, censor=cen)
                              for c in [2,3] for cen in [True,False])
a = pd.DataFrame(abl).round(2)
a.to_csv(f"{OUT}/results_censoring.csv", index=False)
print(a[["confirm","censor","TPR","FPR","delay","regime_acc"]]
      .to_string(index=False))
fig, ax = plt.subplots(figsize=(6,3.8))
labels = [f"{int(r.confirm)}-of-3\n{'censored' if r.censor else 'uncensored'}"
          for _, r in a.iterrows()]
ax.bar(labels, a.TPR, color=["#2b6cb0","#90cdf4","#c53030","#feb2b2"])
ax.set_ylabel("Valid TPR (%)")
ax.set_title("Chart censoring breaks the 3-of-3 rule")
for i, v in enumerate(a.TPR): ax.text(i, v+1.5, f"{v:.1f}", ha="center")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_censoring.png", dpi=200); plt.show()

## EXP3 — Detector comparison: persistent exceedance vs. CUSUM vs. causal CP-PSI

In [ ]:
# === Cell 6: EXP3 ===
det_cfgs = ([("pe", dict(thr=1.75, confirm=2, persist=2))] +
            [("cusum", dict(thr=1.75, confirm=2, persist=2,
                            cusum_k=0.5, cusum_h=h)) for h in [4.0,6.0,8.0]] +
            [("composite", dict(thr=t, confirm=2, persist=2))
             for t in [1.75, 2.0, 2.5]])
res3 = Parallel(n_jobs=N_JOBS)(delayed(run_cell)(detector=d, **kw)
                               for d, kw in det_cfgs)
d3 = pd.DataFrame(res3).round(2)
d3["param"] = ["theta=1.75","h=4","h=6","h=8","theta=1.75","theta=2.0","theta=2.5"]
d3.to_csv(f"{OUT}/results_detectors.csv", index=False)
print(d3[["detector","param","TPR","premature","FPR","delay","lead"]]
      .to_string(index=False))

## EXP4 — Regime classification: horizon sweep and sim-to-real transfer
Rise ratio (single parameter) vs. logistic regression vs. gradient boosting.
All three are trained/evaluated on the synthetic testbed; their zero-shot
transfer to the two real archetypes is assessed in EXP5. Both outcomes of the
sim-to-real comparison are reported as-is in the article.

In [ ]:
# === Cell 7: EXP4 ===
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def gen_labeled(n, seed):
    rng = np.random.default_rng(seed); out = []
    i = 0
    while len(out) < n:
        typ = "endogenous" if i%2==0 else "exogenous"; i += 1
        s = generate_series(typ, rng)
        ign = detect_pe(s, 1.75, 2, persist=2)
        if ign is None or ign < s["onset"]-3: continue
        out.append((s, ign, typ))
    return out

train = gen_labeled(1600, seed=21)
test  = gen_labeled(800,  seed=99)
rows4 = []
for h in [3, 6, 9, 12]:
    Xtr = [causal_features(s,i,h) for s,i,_ in train]
    ytr = [t for *_, t in train]
    Xte = [causal_features(s,i,h) for s,i,_ in test]
    yte = np.array([t for *_, t in test])
    rr = np.array(["exogenous" if f[0]>=0.6 else "endogenous" for f in Xte])
    lr = make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
    gb = GradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
    rows4.append(dict(horizon=h,
                      rise=100*(rr==yte).mean(),
                      logreg=100*(lr.predict(Xte)==yte).mean(),
                      gbm=100*(gb.predict(Xte)==yte).mean()))
    if h == 12: LR12, GB12 = lr, gb   # keep for the real-case cells
r4 = pd.DataFrame(rows4).round(1)
r4.to_csv(f"{OUT}/results_regime_horizon.csv", index=False)
print(r4.to_string(index=False))
fig, ax = plt.subplots(figsize=(6,3.8))
for col, lbl in [("rise","rise-ratio"),("logreg","LogReg"),("gbm","GBM")]:
    ax.plot(r4.horizon, r4[col], marker="o", label=lbl)
ax.set_xlabel("confirmation horizon h (months)")
ax.set_ylabel("regime accuracy (%)"); ax.legend()
ax.set_title("Regime identifiability vs confirmation horizon (synthetic)")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_regime_horizon.png", dpi=200)
plt.show()

## EXP5 — Zero-shot validation on the real archetypes

In [ ]:
# === Cell 8: EXP5 ===
def find(fname):
    hits = glob.glob(f"/kaggle/input/**/{fname}", recursive=True) or \
           glob.glob(f"./**/{fname}", recursive=True)
    assert hits, f"{fname} not found - attach the dataset via Add Input"
    return hits[0]

def load_real(fp, spc, ytc, trc):
    df = pd.read_csv(fp, parse_dates=["final_date"]).set_index("final_date")
    spv = df[spc].fillna(0).to_numpy(float)
    return df.index, dict(sp=np.where(spv>0, np.log10(spv+1), 0.0),
                          yt=np.log10(df[ytc].to_numpy(float)),
                          tr=np.log10(df[trc].to_numpy(float)))

cases = [("Simge - Askin Olayim", "FINAL_ACADEMIC_DATASET_SIMGE.csv",
          ("tr_streams","yt_views","google_search_interest"), "endogenous"),
         ("Kate Bush - RUTH", "FINAL_ACADEMIC_DATASET_KATE_BUSH.csv",
          ("sp_streams","yt_views","trends_interest"), "exogenous")]
rows5, plots = [], []
for name, fname, cols, truth in cases:
    idx, s = load_real(find(fname), *cols)
    ign = detect_pe(s, 1.75, 2, persist=2)
    rr = rise_ratio(s["yt"], ign, 12)
    regime = "exogenous" if rr >= 0.6 else "endogenous"
    f12 = causal_features(s, ign, 12)
    lead = int(np.argmax(s["sp"])) - ign
    rows5.append(dict(case=name, truth=truth,
                      ignition=str(idx[ign].date()),
                      regime_rise=regime, rise_ratio=round(rr,3),
                      regime_logreg=LR12.predict([f12])[0],
                      regime_gbm=GB12.predict([f12])[0],
                      lead_to_peak_mo=lead))
    plots.append((idx, s, ign, name, regime, rr))
r5 = pd.DataFrame(rows5)
r5.to_csv(f"{OUT}/results_real_cases.csv", index=False)
print(r5.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, (idx, s, ign, name, regime, rr) in zip(axes, plots):
    ax.plot(idx, s["yt"], label="YouTube (log10)", lw=1.4)
    ax.plot(idx, s["tr"], label="Trends (log10)", lw=1.4)
    ax.plot(idx, np.where(s["sp"]>0, s["sp"], np.nan),
            label="Spotify (censored)", lw=1.4)
    ax.axvline(idx[ign], color="red", ls="--", lw=1.5,
               label=f"Ignition {idx[ign].strftime('%Y-%m')}")
    ax.set_title(f"{name}\nclassified: {regime} (r={rr:.2f})", fontsize=10)
    ax.legend(fontsize=8, loc="upper left")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_real_cases.png", dpi=200)
plt.show()

## EXP6 — Honest forecasting benchmark (expanding-window, one-step-ahead)

In [ ]:
# === Cell 9: EXP6 ===
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

def expanding_bench(fname, spc, ytc, trc, label, min_train=24):
    df = pd.read_csv(find(fname), parse_dates=["final_date"]).set_index("final_date")
    d = df["2021-01-01":].copy()
    y_raw = np.log1p(d[spc].fillna(0).astype(float))
    m = pd.DataFrame({"y": y_raw})
    for l in [1,2,3]: m[f"yt{l}"] = np.log1p(d[ytc]).shift(l)
    for l in [1,2]: m[f"tr{l}"] = np.log1p(d[trc]).shift(l)
    m["sp1"] = m["y"].shift(1)
    m = m.dropna()
    X, y = m.drop(columns="y"), m["y"]
    preds = {"SVR": [], "RF": [], "Ridge": [], "Naive": []}; idxs = []
    for t in range(min_train, len(m)):
        Xtr, ytr, xt = X.iloc[:t], y.iloc[:t], X.iloc[[t]]
        preds["SVR"].append(SVR(kernel="rbf", C=8.0, epsilon=0.1)
                            .fit(Xtr, ytr).predict(xt)[0])
        preds["RF"].append(RandomForestRegressor(150, max_depth=6,
                           random_state=42).fit(Xtr, ytr).predict(xt)[0])
        preds["Ridge"].append(Ridge().fit(Xtr, ytr).predict(xt)[0])
        preds["Naive"].append(y.iloc[t-1]); idxs.append(t)
    yt_ = y.iloc[idxs]
    return [dict(case=label, model=k, R2=round(r2_score(yt_, v),3),
                 MAE=round(mean_absolute_error(yt_, v),3))
            for k, v in preds.items()]

bench = (expanding_bench("FINAL_ACADEMIC_DATASET_SIMGE.csv","tr_streams",
                         "yt_views","google_search_interest","Simge (endo)") +
         expanding_bench("FINAL_ACADEMIC_DATASET_KATE_BUSH.csv","sp_streams",
                         "yt_views","trends_interest","Kate Bush (exo)"))
b = pd.DataFrame(bench)
b.to_csv(f"{OUT}/results_forecast_bench.csv", index=False)
print(b.to_string(index=False))

## EXP4b + EXP7 — delta-jump baseline and multi-seed statistics
- **EXP4b:** The single-jump delta rule of the original IVPM (2.0 sigma) and a
  fair variant with the cut tuned on the training set, evaluated under the same
  test conditions as rise-ratio / LogReg / GBM.
- **EXP7:** Replication over 5 seeds; mean +/- sd, 95% CI (t distribution, n=5),
  within-run paired **McNemar** tests, and across-seed **Wilcoxon**
  signed-rank tests.

In [ ]:
# === Cell 11: EXP4b tanimlar — delta-sicrama siniflandiricisi ===
from scipy.stats import binomtest, wilcoxon, t as tdist

SEEDS = [7, 17, 27, 37, 47]
N_TRAIN, N_TEST = 1600, 800

def delta_jump(s, ign):
    """Maximum causal z-jump at ignition (the original IVPM statistic)."""
    best = 0.0
    for x, cens in [(s["yt"],False),(s["tr"],False),(s["sp"],True)]:
        p = x[:ign]; p = p[p>0] if cens else p
        if len(p) > 5:
            best = max(best, abs(x[ign]-p[-1])/(p.std() or 1e-6))
    return best

def ci95(v):
    v = np.asarray(v, float); n = len(v)
    h = tdist.ppf(0.975, n-1)*v.std(ddof=1)/np.sqrt(n)
    return v.mean(), v.std(ddof=1), h

def mcnemar_p(correct_a, correct_b):
    b = int(np.sum(correct_a & ~correct_b))
    c = int(np.sum(~correct_a & correct_b))
    return (binomtest(min(b,c), b+c).pvalue if b+c>0 else 1.0), b, c
print("EXP4b/EXP7 definitions ready | seeds:", SEEDS)

In [ ]:
# === Cell 12: EXP7a — multi-seed operating points ===
cfg7a = ([(th,2,2,True) for th in [1.25,1.5,1.75,2.0,2.25,2.5]] +
         [(1.75,1,2,True),(1.75,3,2,True),(1.75,2,2,False),(1.75,3,2,False)])
jobs = [(th,c,p,cen,sd) for (th,c,p,cen) in cfg7a for sd in SEEDS]
res7a = Parallel(n_jobs=N_JOBS)(
    delayed(run_cell)(th, c, p, censor=cen, seed=sd) for th,c,p,cen,sd in jobs)
sg = pd.DataFrame(res7a)
sg["seed"] = [sd for *_, sd in jobs]
sg.to_csv(f"{OUT}/results_seed_grid.csv", index=False)

def agg(gdf):
    out = {}
    for m in ["TPR","FPR","premature","delay","lead","regime_acc"]:
        mu, sdv, h = ci95(gdf[m].values)
        out[m] = f"{mu:.2f} ± {sdv:.2f} (±{h:.2f})"
    return pd.Series(out)
stats7a = (sg.groupby(["thr","confirm","persist","censor"])
             .apply(agg).reset_index())
stats7a.to_csv(f"{OUT}/results_seed_grid_stats.csv", index=False)
print("Multi-seed statistics [mean ± sd (± 95% CI)]:")
print(stats7a.to_string(index=False))

In [ ]:
# === Cell 13: EXP7b — regime classification: 5 seeds + EXP4b + McNemar ===
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def regime_seed_run(sd):
    train = gen_labeled(N_TRAIN, seed=sd)
    test  = gen_labeled(N_TEST,  seed=sd+1000)
    yte = np.array([t for *_, t in test])
    out = dict(seed=sd)
    # delta baseline (independent of h)
    dj_tr = np.array([delta_jump(s,i) for s,i,_ in train])
    ytr = np.array([t for *_, t in train])
    dj_te = np.array([delta_jump(s,i) for s,i,_ in test])
    pred_d20 = np.where(dj_te >= 2.0, "exogenous", "endogenous")
    cuts = np.arange(0.5, 4.01, 0.25)
    accs = [( (np.where(dj_tr>=c,"exogenous","endogenous")==ytr).mean(), c)
            for c in cuts]
    best_cut = max(accs)[1]
    pred_dtn = np.where(dj_te >= best_cut, "exogenous", "endogenous")
    out["delta20"] = 100*(pred_d20==yte).mean()
    out["delta_tuned"] = 100*(pred_dtn==yte).mean()
    out["delta_best_cut"] = best_cut
    corr = {"delta20": pred_d20==yte, "delta_tuned": pred_dtn==yte}
    for h in [3, 6, 12]:
        Xtr = [causal_features(s,i,h) for s,i,_ in train]
        Xte = [causal_features(s,i,h) for s,i,_ in test]
        pr = np.array(["exogenous" if f[0]>=0.6 else "endogenous" for f in Xte])
        lr = make_pipeline(StandardScaler(),
                           LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
        gb = GradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
        out[f"rise_h{h}"]   = 100*(pr==yte).mean()
        out[f"logreg_h{h}"] = 100*(lr.predict(Xte)==yte).mean()
        out[f"gbm_h{h}"]    = 100*(gb.predict(Xte)==yte).mean()
        if h == 12:
            corr["rise"] = pr==yte
            corr["gbm"]  = gb.predict(Xte)==yte
    # within-run paired McNemar tests (h=12)
    for a, b in [("rise","delta20"), ("rise","delta_tuned"), ("gbm","rise")]:
        p, nb_, nc_ = mcnemar_p(corr[a], corr[b])
        out[f"mcnemar_{a}_vs_{b}_p"] = p
    return out

res7b = Parallel(n_jobs=min(N_JOBS, len(SEEDS)))(
    delayed(regime_seed_run)(sd) for sd in SEEDS)
sr = pd.DataFrame(res7b)
sr.to_csv(f"{OUT}/results_seed_regime.csv", index=False)
print(sr.round(2).to_string(index=False))

print("\n--- Summary [mean ± sd (± 95% CI)] ---")
for col in ["rise_h3","rise_h6","rise_h12","logreg_h12","gbm_h12",
            "delta20","delta_tuned"]:
    mu, sdv, h = ci95(sr[col].values)
    print(f"  {col:12s}: {mu:5.1f} ± {sdv:4.1f} (±{h:4.1f})")

print("\n--- Across-seed Wilcoxon (n=5; attainable minimum p = 0.0625) ---")
for a, b in [("rise_h12","delta20"), ("rise_h12","delta_tuned"),
             ("gbm_h12","rise_h12"), ("rise_h12","rise_h3")]:
    try:
        wp = wilcoxon(sr[a], sr[b]).pvalue
    except ValueError:
        wp = float("nan")
    print(f"  {a} vs {b}: diff={sr[a].mean()-sr[b].mean():+.1f} points, p={wp:.4f}")
print("\nMcNemar (within-run, h=12) median p-values:")
for c in [c for c in sr.columns if c.startswith("mcnemar")]:
    print(f"  {c}: median p = {sr[c].median():.2e}")

In [ ]:
# === Cell 14: EXP7 figures ===
# (a) operating curve with error bars
sw = sg[(sg.confirm==2)&(sg.persist==2)&(sg.censor==True)]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, m, yl in zip(axes, ["TPR","FPR","delay"],
                     ["Valid TPR (%)","False-alarm rate (%)","Delay (mo)"]):
    st = sw.groupby("thr")[m].agg(["mean","std"]).reset_index()
    ax.errorbar(st.thr, st["mean"], yerr=st["std"], marker="o", capsize=4)
    ax.set_xlabel(r"threshold $\theta$ ($\sigma$)"); ax.set_ylabel(yl)
fig.suptitle("2-of-3, persistence=2: mean ± std over 5 seeds")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_operating_errorbars.png", dpi=200)
plt.show()

# (b) seed distributions of the regime classifiers (h=12)
fig, ax = plt.subplots(figsize=(7.5, 4))
cols = ["rise_h12","logreg_h12","gbm_h12","delta20","delta_tuned"]
lbl = ["rise-ratio","LogReg","GBM",r"$\delta$=2.0 (IVPM v1)",r"$\delta$ tuned"]
bp = ax.boxplot([sr[c] for c in cols], labels=lbl, patch_artist=True)
for pc, cc in zip(bp["boxes"],
                  ["#2f855a","#2b6cb0","#6b46c1","#a0aec0","#718096"]):
    pc.set_facecolor(cc); pc.set_alpha(0.6)
ax.set_ylabel("Regime accuracy (%)  [synthetic, h=12]")
ax.axhline(50, color="k", ls=":", lw=1)
ax.set_title("Classifier robustness across 5 seeds")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_regime_seed_box.png", dpi=200)
plt.show()

# (c) horizon curves with confidence bands
fig, ax = plt.subplots(figsize=(6.5, 4))
for fam, cc in [("rise","#2f855a"),("logreg","#2b6cb0"),("gbm","#6b46c1")]:
    hs, mus, err = [3,6,12], [], []
    for h in [3,6,12]:
        mu, sdv, hw = ci95(sr[f"{fam}_h{h}"].values)
        mus.append(mu); err.append(hw)
    ax.errorbar(hs, mus, yerr=err, marker="o", capsize=4, label=fam, color=cc)
mu, sdv, hw = ci95(sr["delta20"].values)
ax.axhspan(mu-hw, mu+hw, color="gray", alpha=0.25)
ax.axhline(mu, color="gray", ls="--", lw=1.2, label=r"$\delta$=2.0 (v1)")
ax.set_xlabel("confirmation horizon h (months)")
ax.set_ylabel("regime accuracy (%)"); ax.legend()
ax.set_title("Regime identifiability: mean ± 95% CI (5 seeds)")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_regime_horizon_ci.png", dpi=200)
plt.show()

# (d) TPR-FPR scatter (all detector families)
fig, ax = plt.subplots(figsize=(6.5, 4.2))
pe_pts = sw.groupby("thr")[["FPR","TPR"]].mean().reset_index()
ax.plot(pe_pts.FPR, pe_pts.TPR, "o-", color="#2f855a",
        label="persistent exceedance (seed-mean)")
for _, r in pe_pts.iterrows():
    ax.annotate(f"{r.thr:g}", (r.FPR, r.TPR), fontsize=8,
                xytext=(4,-4), textcoords="offset points")
dcu = d3[d3.detector=="cusum"]
ax.plot(dcu.FPR, dcu.TPR, "s", color="#c53030", label="CUSUM")
dco = d3[d3.detector=="composite"]
ax.plot(dco.FPR, dco.TPR, "^", color="#2b6cb0", label="causal CP-PSI")
ax.set_xlabel("False-alarm rate (%)"); ax.set_ylabel("Valid TPR (%)")
ax.set_xscale("symlog", linthresh=1)
ax.legend(); ax.set_title("Detector families in the 2-of-3 architecture")
fig.tight_layout(); fig.savefig(f"{OUT}/fig_pareto.png", dpi=200)
plt.show()

In [ ]:
# === Cell 15: delta rule on the real archetypes + SUMMARY v2 + zip ===
rows_rd = []
for name, fname, colsx, truth in cases:
    idx, s = load_real(find(fname), *colsx)
    ign = detect_pe(s, 1.75, 2, persist=2)
    dj = delta_jump(s, ign)
    rows_rd.append(dict(case=name, truth=truth, delta_jump=round(dj,3),
                        regime_delta20="exogenous" if dj>=2.0 else "endogenous"))
rd = pd.DataFrame(rows_rd)
rd.to_csv(f"{OUT}/results_real_delta.csv", index=False)

print("="*68)
print("HEADLINE RESULTS v2 (multi-seed)")
sel = sg[(sg.thr==1.75)&(sg.confirm==2)&(sg.persist==2)&(sg.censor==True)]
for m in ["TPR","FPR","delay"]:
    mu, sdv, h = ci95(sel[m].values)
    print(f"  Selected point {m}: {mu:.2f} ± {sdv:.2f} (95% CI ±{h:.2f})")
c33 = sg[(sg.confirm==3)&(sg.censor==True)]; c33u = sg[(sg.confirm==3)&(sg.censor==False)]
print(f"  3-of-3 censored TPR: {c33.TPR.mean():.2f} ± {c33.TPR.std(ddof=1):.2f} "
      f"vs uncensored {c33u.TPR.mean():.2f} ± {c33u.TPR.std(ddof=1):.2f}")
mu_r, _, h_r = ci95(sr["rise_h12"]); mu_d, _, h_d = ci95(sr["delta20"])
print(f"  Regime (h=12): rise={mu_r:.1f}±{h_r:.1f} vs delta2.0={mu_d:.1f}±{h_d:.1f} "
      f"(McNemar median p={sr['mcnemar_rise_vs_delta20_p'].median():.1e})")
print("  Delta rule on the real archetypes:"); print(rd.to_string(index=False))
print("  Rise/ML on the real archetypes:"); print(r5.to_string(index=False))
zp = f"{OUT}/ivpm_ew_results_v2.zip"
with zipfile.ZipFile(zp, "w") as z:
    for f in glob.glob(f"{OUT}/results_*.csv") + glob.glob(f"{OUT}/fig_*.png"):
        z.write(f, os.path.basename(f))
print("Bundled:", zp)

## Collecting the outputs (v2)
All tables and figures used in the article are reproduced by this notebook and
bundled into `ivpm_ew_results_v2.zip`, together with the console summary block
(`HEADLINE RESULTS v2`). Key files: `results_seed_grid_stats.csv` (operating
points with confidence intervals), `results_seed_regime.csv` (per-seed
classifier accuracies and McNemar p-values), and `results_real_delta.csv`
(the delta rule's decisions on the real archetypes). Runtime: about 20-35
minutes on top of EXP1-EXP6; no GPU is used.